In [ ]:
from langchain_core.tools import tool

In [ ]:
@tool
def multiply_int(a:int , b:int) -> int:
    """Function to generate multiplication of 2 input variable"""
    return a*b


In [ ]:
multi = multiply_int.invoke({"a": 5, "b": 10})
print(multi)

In [ ]:
print(multiply_int.name)
print(multiply_int.description)
print(multiply_int.args)

In [ ]:
multiply_int.args_schema.model_json_schema()

Structure Tools and pydantic

In [ ]:
from langchain.tools import StructuredTool
from pydantic import BaseModel,Field

In [ ]:
class MultiplyInput(BaseModel):
    a: int = Field(required=True,description="First number to add")
    b: int = Field(required=True,description="Second number to add")

def multiply_func(a:int , b:int) -> int:
    return a*b


In [ ]:
multiply_model = StructuredTool.from_function(func=multiply_func,description="Multiply two numbers",args_schema=MultiplyInput,name="multiply")

result = multiply_model.invoke({"a":5,"b":10})

print(result)

In [ ]:
print(multiply_model.name)
print(multiply_model.description)
print(multiply_model.args)

In [ ]:
multiply_model.args_schema.model_json_schema()

In [ ]:
from langchain.tools import BaseTool
from typing import Type
from pydantic import BaseModel, Field

In [ ]:
class MultiplyInput(BaseModel):
    a: int = Field(required= True, description="first number to add")
    b: int = Field(required=True, description="second number to add")
    
class MultiplyTool(BaseTool):
    name: str = "Multiply"
    description: str = "get multiply of 2 numbers"
    
    args_schema: Type[BaseModel] = MultiplyInput
    
    def _run(self, a:int ,b:int)->int:
        return a*b

In [ ]:
multiply_tools = MultiplyTool()

result = multiply_tools.invoke({"a":10,"b":10})

print(result)


In [ ]:
multiply_tools.args_schema.model_json_schema()

ToolKits

In [ ]:
from langchain.tools import tool


@tool
def add(a:int,b:int) -> int:
    """Addition of 2 numbers"""
    return a + b

@tool
def multiply(a:int , b:int) -> int:
    """Multiplication of 2 numbers"""
    return a * b

class MathtoolKit():
    def get_tools(self):
        return [add,multiply]

In [ ]:
toolkit = MathtoolKit()
tools = toolkit.get_tools()

In [ ]:
for tool in tools:
    print(tool.name)
    print(tool.description)
    print(tool.args)

In [ ]:
# Invoke by accessing specific tools from the list
add_tool      = tools[0]  # add
multiply_tool = tools[1]  # multiply

# Call with a dictionary of arguments
result_add      = add_tool.invoke({"a": 5, "b": 3})
result_multiply = multiply_tool.invoke({"a": 5, "b": 3})

print(result_add)       # 8
print(result_multiply)  # 15

In [ ]:
# Build a name → tool map
tool_map = {t.name: t for t in tools}
print(tool_map)
# Invoke by name
result = tool_map["add"].invoke({"a": 10, "b": 20})
print(result)  # 30

result = tool_map["multiply"].invoke({"a": 4, "b": 7})
print(result)  # 28

In [ ]:
inputs = {"a": 6, "b": 4}

for t in tools:
    result = t.invoke(inputs)
    print(f"{t.name}({inputs}) = {result}")

Tool Binding

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.tools import tool

@tool
def multiply(a: int , b: int) -> int:
    "Function will provide product of 2 provided numbers"
    return a * b

# result = multiply.invoke({"a":10 ,"b" : 10})
# print(result)

llm = ChatOpenAI()

llm_with_tools = llm.bind_tools([multiply])

llm_with_tools.invoke('Hi, How are you?')

In [ ]:
from langchain_core.messages import HumanMessage
query = HumanMessage('What is the product of 10 & 10')
messages = [query]
print(messages)

In [ ]:
result = llm_with_tools.invoke(messages)
print([result])

In [ ]:
messages.append(result)

In [ ]:
print(messages)

In [ ]:
result.tool_calls[0]

In [ ]:
tool_message = multiply.invoke(result.tool_calls[0])

In [ ]:
print(tool_message)

In [ ]:
messages.append(tool_message)

In [ ]:
print(messages)

In [ ]:
final_result = llm.invoke(messages)

In [ ]:
print(final_result.content)

Real Time Currency Conversation Tool

In [1]:
from langchain_core.tools import tool,InjectedToolArg
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI
import requests
from typing import Annotated

@tool
def getConversationRatio(BaseCurrency: str,TargetCurrency: str) -> float:
    """Function will fetch only fetch realtime conversation rate between base currency and target currency"""
    url = f'https://v6.exchangerate-api.com/v6/1412473e690df8a142e5bd27/pair/{BaseCurrency}/{TargetCurrency}'
    response = requests.get(url).json()
    conversation_rate = response.get("conversion_rate")
    print(f"Conversation rate between {BaseCurrency} and {TargetCurrency} is {conversation_rate}")
    return conversation_rate

@tool
def convert(base_currency_value: float,conversation_rate: float) -> float:
    """Given real time conversation rate , this function will calculate currency value from given base currency value"""

    return conversation_rate * base_currency_value

d:\VS\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [2]:
llm = ChatOpenAI()
llm_with_tools = llm.bind_tools([getConversationRatio,convert])
print(llm_with_tools)

query = HumanMessage("what is the the current conversation rate between USD and INR. Based on conversation rate convert 80 USD to INR")
messages = []
messages.append(query)
print(messages)

bound=ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x000001EBED5797F0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001EBED57A270>, root_client=<openai.OpenAI object at 0x000001EBECCC4D70>, root_async_client=<openai.AsyncOpenAI object at 0x000001EBED579FD0>, model_kwargs={}, openai_api_key=SecretStr('**********'), stream_usage=True) kwargs={'tools': [{'type': 'function', 'function': {'name': 'getConversationRatio', 'description': 'Function will fetch only fetch realtime conversation rate between base currency and target currency', 'parameters': {'properties': {'BaseCurrency': {'type': 'string'}, 'TargetCurrency': {'type': 'string'}}, 'required': ['BaseCurrency', 'TargetCurrency'], 'type': 'object'}}}, {'type': 'function', 'function': {'name': 'convert', 'description': 'Given real time conversation rate , this function will calculate currency value from given base currency value', 'parameters': {'p

In [3]:
ai_result = llm_with_tools.invoke(messages)
messages.append(ai_result)
print(messages)
print(ai_result.tool_calls)

[HumanMessage(content='what is the the current conversation rate between USD and INR. Based on conversation rate convert 80 USD to INR', additional_kwargs={}, response_metadata={}), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_5j09VEnSY7COFkfr9O7SpOhl', 'function': {'arguments': '{"BaseCurrency":"USD","TargetCurrency":"INR"}', 'name': 'getConversationRatio'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 123, 'total_tokens': 145, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DSH1IxslTOi7UzBIsCnYro3iPZH82', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d6bda-85c7-7f40-8c2c-2263d95f0fa2-0', tool_calls=[{'name': 'g

In [4]:
for tool_call in ai_result.tool_calls:
    if tool_call.get("name") == "getConversationRatio":
        response1 = getConversationRatio.invoke(tool_call)
        messages.append(response1)
        currency_conv_rate = response1.content
        print(currency_conv_rate)
    if tool_call.get("name") == "convert":
        tool_call.get("args")["conversation_rate"] = currency_conv_rate
        print(tool_call)
        response2 = convert.invoke(tool_call)
        messages.append(response2)

Conversation rate between USD and INR is 93.1537
93.1537


In [5]:
print(messages)

[HumanMessage(content='what is the the current conversation rate between USD and INR. Based on conversation rate convert 80 USD to INR', additional_kwargs={}, response_metadata={}), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_5j09VEnSY7COFkfr9O7SpOhl', 'function': {'arguments': '{"BaseCurrency":"USD","TargetCurrency":"INR"}', 'name': 'getConversationRatio'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 123, 'total_tokens': 145, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DSH1IxslTOi7UzBIsCnYro3iPZH82', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d6bda-85c7-7f40-8c2c-2263d95f0fa2-0', tool_calls=[{'name': 'g

In [6]:
ai_result2 = llm_with_tools.invoke(messages)
messages.append(ai_result2)
print(messages)
print(ai_result2.tool_calls)

[HumanMessage(content='what is the the current conversation rate between USD and INR. Based on conversation rate convert 80 USD to INR', additional_kwargs={}, response_metadata={}), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_5j09VEnSY7COFkfr9O7SpOhl', 'function': {'arguments': '{"BaseCurrency":"USD","TargetCurrency":"INR"}', 'name': 'getConversationRatio'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 123, 'total_tokens': 145, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DSH1IxslTOi7UzBIsCnYro3iPZH82', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d6bda-85c7-7f40-8c2c-2263d95f0fa2-0', tool_calls=[{'name': 'g

In [7]:
for tool_call in ai_result2.tool_calls:
    if tool_call.get("name") == "getConversationRatio":
        response1 = getConversationRatio.invoke(tool_call)
        messages.append(response1)
        currency_conv_rate = response1.content
        print(currency_conv_rate)
    if tool_call.get("name") == "convert":
        #tool_call.get("args")["conversation_rate"] = currency_conv_rate
        print(tool_call)
        response2 = convert.invoke(tool_call)
        messages.append(response2)

{'name': 'convert', 'args': {'base_currency_value': 80, 'conversation_rate': 93.1537}, 'id': 'call_hWoFLFcRcrWXH0ELtMZfSOiX', 'type': 'tool_call'}


In [8]:
print(messages)

[HumanMessage(content='what is the the current conversation rate between USD and INR. Based on conversation rate convert 80 USD to INR', additional_kwargs={}, response_metadata={}), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_5j09VEnSY7COFkfr9O7SpOhl', 'function': {'arguments': '{"BaseCurrency":"USD","TargetCurrency":"INR"}', 'name': 'getConversationRatio'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 123, 'total_tokens': 145, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DSH1IxslTOi7UzBIsCnYro3iPZH82', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d6bda-85c7-7f40-8c2c-2263d95f0fa2-0', tool_calls=[{'name': 'g

In [9]:
final_llm_response = llm.invoke(messages)
print(final_llm_response)
print(final_llm_response.content)

content='The current conversion rate between USD and INR is 1 USD to 93.1537 INR. \n\nWhen you convert 80 USD, it equals 7452.296 INR.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 40, 'prompt_tokens': 100, 'total_tokens': 140, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DSH1y2XkvVotgyAOOCbclrqaeBgCR', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019d6bdb-2c92-7db1-aff1-1187d8440278-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 100, 'output_tokens': 40, 'total_tokens': 140, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}
The current conversion rate between USD and INR is 1 USD to 93